In [2]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [5]:
df = pd.read_csv('RegressionData.csv')
df.head()

,POLY_ID,AREAKEY,MEDHVAL,PCTBACHMOR,MEDHHINC,PCTVACANT,PCTSINGLES,NBELPOV100,Police.Unit
0,1,4.210000e+11,225000,64.4737,49107,13.2075,11.3208,0,A
1,2,4.210000e+11,181900,78.7805,46304,0.0000,0.0000,25,A
2,3,4.210000e+11,225000,45.0000,26875,0.0000,42.8571,14,A
3,4,4.210000e+11,113600,64.3564,49375,7.7500,0.0000,70,A
4,5,4.210000e+11,126200,13.0690,9432,9.7387,0.0000,473,A


In [12]:
target = 'MEDHVAL'
features = ['PCTBACHMOR', 'MEDHHINC', 'PCTVACANT', 'PCTSINGLES']

df = df.dropna(subset=features + [target])
X = df[features].values
y = df[target].values

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
x_scaler = StandardScaler().fit(X_train)
X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)

In [16]:
y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
y_train_s = y_scaler.transform(y_train.reshape(-1, 1)).ravel()

In [20]:
nn_pred = y_scaler.inverse_transform(nn.predict(X_test_s).reshape(-1, 1)).ravel()

In [21]:
lr = LinearRegression().fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)

In [22]:
print(f"{'Model':<22}{'R^2':>8}{'MAE ($)':>14}")
print(f"{'Linear regression':<22}{r2_score(y_test, lr_pred):>8.3f}{mean_absolute_error(y_test, lr_pred):>14,.0f}")
print(f"{'Neural network':<22}{r2_score(y_test, nn_pred):>8.3f}{mean_absolute_error(y_test, nn_pred):>14,.0f}")

Model                      R^2       MAE ($)
Linear regression        0.559        17,355
Neural network           0.497        16,830


In [27]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.neural_network import MLPRegressor

from sklearn.compose import TransformedTargetRegressor


models = {
    "Linear regression": LinearRegression(),
    "Ridge": RidgeCV(),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient boosting": HistGradientBoostingRegressor(random_state=42),
    "Neural network": MLPRegressor(hidden_layer_sizes=(16, 8), max_iter=2000, early_stopping=True, random_state=42),
}

nn = TransformedTargetRegressor(
    regressor=MLPRegressor(hidden_layer_sizes=(16, 8), max_iter=2000,
                           early_stopping=True, random_state=42),
    transformer=StandardScaler(),
)
models["Neural network"] = nn

for name, model in models.items():
    pipe = make_pipeline(StandardScaler(), model)
    scores = -cross_val_score(pipe, X, y, cv=5, scoring="neg_root_mean_squared_error")
    print(f"{name:<20} RMSE = ${scores.mean():,.0f} (± {scores.std():,.0f})")

Linear regression    RMSE = $37,775 (± 17,040)
Ridge                RMSE = $37,736 (± 17,063)
Random forest        RMSE = $39,276 (± 17,934)
Gradient boosting    RMSE = $41,441 (± 16,961)
Neural network       RMSE = $39,595 (± 16,811)
